#  Single-nuclei Pseudobulk Preprocessing (RNA-seq and ATAC-seq)


## Description

Single-nuclei pseudobulk preprocessing for RNA-seq and ATAC-seq. Aggregates per-nucleus counts into per-sample pseudobulk matrices, harmonizes sample IDs, and regresses out technical covariates to produce QTL-ready phenotype values.

The pipeline has three stages:
- `pseudobulk_counts` -- aggregate a Seurat object into a raw pseudobulk count matrix for one cell type.
- `sampleid_mapping` -- remap `individualID` headers to standardized `sampleid` across metadata and count matrices.
- `pseudobulk_qc` -- filter, TMM-normalize, fit a technical-covariate model, and write residuals.

(`phenotype_formatting` reformats residuals into a BED for snATAC-seq caQTL mapping.)

**Timing:** ~5-10 min on typical compute infrastructure.

## Input Files

The toy example uses single-nuclei RNA-seq data for the `MIC` (microglia) cell type under `input/snrnaseq/`.

| File | Description |
|------|-------------|
| `protocol_example.snrnaseq.seurat_MIC.rds` | Seurat object (MIC cells) with per-nucleus counts (input to `pseudobulk_counts`) |
| `protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz` | Pre-aggregated pseudobulk count matrix (genes x samples) |
| `protocol_example.snrnaseq.metadata_MIC.csv` | Sample-level metadata (sampleid, batch, nuclei counts) |
| `protocol_example.snrnaseq.tech_vars_MIC.csv` | Technical covariates keyed by `sampleid` |
| `protocol_example.snrnaseq.id_map.csv` | individualID -> sampleid map (input to `sampleid_mapping`) |
| `atac_residuals/MIC/protocol_example.snrnaseq.MIC_residuals.txt` | Peak residuals with `chr-start-end` IDs (input to `phenotype_formatting`) |

## Step 1: Pseudobulk Count Matrix Generation

Aggregates single-nuclei counts into pseudobulk count matrices per cell type from Seurat objects.

> This step is upstream of `sampleid_mapping` and `pseudobulk_qc`. Output feeds directly into the existing preprocessing pipeline.

### Input

| File | Description |
|------|-------------|
| `celltyped_seuratobj{i}.rds` | Seurat objects with `celltype` and `sample` annotations in `meta.data` |

### Process

1. Load each Seurat object and subset to target cell type — skips objects where cell type is not present
2. Merge all subsets across objects and join layers
3. Aggregate raw counts by sample (`AggregateExpression`)
4. Filter out samples with fewer than `min_cells` cells (default: 10)
5. Strip Ensembl version suffixes from gene IDs (`ENSG00000000010.1` → `ENSG00000000010`)
6. Save as `pseudobulk_counts_{celltype}.csv.gz` — raw counts only, normalization handled downstream in `pseudobulk_qc`

> **GLU cell type**: Due to its large size, process in two batches (files 1–6 and 7–11) and pass separately.

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `seurat_files` | *required* | One or more Seurat `.rds` files |
| `output_dir` | *required* | Output directory for count matrix |
| `celltype` | `MIC` | Cell type to extract (must match `celltype` column in `meta.data`) |
| `min_cells` | `10` | Minimum number of cells per sample to retain |

### Output

| File | Description |
|------|-------------|
| `pseudobulk_counts_{celltype}.csv.gz` | Raw pseudobulk count matrix (genes × samples) |


**Timing:** 10–30 min per cell type depending on object size

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb pseudobulk_counts \
    --seurat-files input/snrnaseq/protocol_example.snrnaseq.seurat_MIC.rds \
    --celltype MIC \
    --output-dir output/snrna_seq


## Step 2: Sample ID Mapping

Maps original sample identifiers (`individualID`) to standardized sample IDs (`sampleid`)
across metadata and count matrix files.

### Input

| File | Description |
|------|-------------|
| `rosmap_sample_mapping_data.csv` | Mapping reference: `individualID → sampleid` |
| `metadata_{celltype}.csv` | Per-cell-type sample metadata |
| `pseudobulk_peaks_counts_{celltype}.csv.gz` *(snATAC-seq)* | Per-cell-type peak count matrices |
| `pseudobulk_counts_{celltype}.csv.gz` *(snRNA-seq)* | Per-cell-type gene count matrices |

### Process

**Part 1 — Metadata files**

For each metadata file:
1. Look up each `individualID` in the mapping reference
2. Assign `sampleid` — falls back to `individualID` if no mapping found
3. Reorder columns: `sampleid` first, then `individualID`, then the rest
4. Save updated file

**Part 2 — Count matrix files**

For each count file:
1. Extract the header row (column names only)
2. Keep the first column (peak or gene IDs) unchanged
3. Map remaining column names (`individualID` → `sampleid`) where mapping exists, otherwise keep original
4. Write new header and stream data rows unchanged
5. Recompress with gzip

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `map_file` | *required* | CSV with `individualID` → `sampleid` mapping |
| `meta_files` | *required* | Metadata CSV files to remap |
| `count_files` | *required* | Count CSV.gz files to remap |
| `output_dir` | *required* | Parent output directory; writes to `{output_dir}/1_files_with_sampleid/` |

### Output

Output directory: `{output_dir}/1_files_with_sampleid/`

| File | Description |
|------|-------------|
| `metadata_{celltype}.csv` | Metadata with `sampleid` column prepended |
| `pseudobulk_peaks_counts_{celltype}.csv.gz` *(snATAC-seq)* | Count matrices with mapped column headers |
| `pseudobulk_counts_{celltype}.csv.gz` *(snRNA-seq)* | Count matrices with mapped column headers |


**Timing:** < 1 min

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb sampleid_mapping \
    --map-file input/snrnaseq/protocol_example.snrnaseq.id_map.csv \
    --meta-files input/snrnaseq/protocol_example.snrnaseq.metadata_MIC.csv \
    --output-dir output/snrna_seq


## Step 2: Pseudobulk QC

Regresses out technical covariates for downstream QTL analysis. Works for both snATAC-seq and snRNA-seq.

### Input

| File | Description |
|------|-------------|
| `metadata_{celltype}.csv` | Sample-level metadata (nuclei counts, batch info) |
| `pseudobulk_*counts_{celltype}.csv.gz` | Pseudobulk count matrix |
| `tech_vars.csv` | Technical covariates (sampleid + tech var columns, pre-processed) |
| `hg38-blacklist.v2.bed.gz` *(snATAC-seq, optional)* | Blacklisted genomic regions |

### Process

1. Load count matrix and auto-detect modality (snATAC-seq vs snRNA-seq)
2. ***(Optional)*** Filter to specific genomic regions (snATAC-seq) or gene list (snRNA-seq)
3. Load metadata; filter samples with fewer than `min_nuclei` nuclei (default: 20)
4. Align samples between metadata and count matrix
5. ***(Optional)*** Filter blacklisted genomic regions (snATAC-seq only)
6. Merge tech vars from `tech_vars_file` by `sampleid` 
7. Drop samples with NA in any tech var
8. Apply expression filtering (`filterByExpr`):
   - `min_count = 5`: minimum reads in at least one sample
   - `min_total_count = 15`: minimum total reads across all samples
   - `min_prop = 0.1`: feature expressed in ≥10% of samples
9. TMM normalization
10. ***(Optional)*** Batch correction on `sequencingBatch`:
    - `limma::removeBatchEffect` (default)
    - `ComBat` (on log-CPM)
11. Add `sequencingBatch` and `Library` to model if present and multi-level
12. Fit linear model (`voom` + `lmFit` + `eBayes`) with **tech vars + batch vars only** 
13. Compute `offset + residuals` as final adjusted values:
    - `offset`: intercept + batch effects at reference level
    - `residuals`: variation after removing technical effects; biological signal retained
14. ***(Optional)*** Quantile normalization of final values

**Model formula:**
```
~ {tech_vars} + [sequencingBatch] + [Library]
```
> `sequencingBatch` and `Library` included only if present and have more than one level.
> Biological variables (`pmi`, `study`, `msex`, `age_death` etc.) are **not** included — they should not be regressed out as they may be associated with genotype.

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `meta_files` | *required* | Metadata CSV files (one per cell type) |
| `count_files` | *required* | Count CSV.gz files (one per cell type, same order as `meta_files`) |
| `output_dir` | *required* | Parent output directory; writes to `{output_dir}/2_residuals/{ct}/` |
| `tech_vars_file` | *required* | CSV with `sampleid` + tech var columns |
| `blacklist_file` | `''` | Genomic blacklist BED file (snATAC-seq only) |
| `regions` | `''` | Comma-separated genomic regions e.g. `chr7:28000000-28300000` (snATAC-seq) |
| `gene_list` | `''` | Comma-separated gene IDs e.g. `ENSG00000000010` (snRNA-seq) |
| `batch_correction` | `FALSE` | Apply batch correction (`TRUE`/`FALSE`) |
| `batch_method` | `limma` | Batch correction method (`limma` or `combat`) |
| `quant_norm` | `FALSE` | Apply quantile normalization after residuals |
| `min_count` | `5` | Min reads in at least one sample |
| `min_total_count` | `15` | Min total reads across all samples |
| `min_prop` | `0.1` | Min proportion of samples with expression |
| `min_nuclei` | `20` | Min nuclei per sample |

### Output

Output directory: `{output_dir}/2_residuals/{celltype}/`

| File | Description |
|------|-------------|
| `{celltype}_residuals.txt` | Tech-covariate-adjusted values (log2-CPM) |
| `{celltype}_residuals_qn.txt` | Quantile-normalized adjusted values *(if `quant_norm=TRUE`)* |
| `{celltype}_results.rds` | Full results: DGEList, fit, offset, residuals, design, parameters |
| `{celltype}_filtered_raw_counts.txt` | Filtered raw counts before normalization |

**Timing:** < 5 min per cell type

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb pseudobulk_qc \
    --meta-files input/snrnaseq/protocol_example.snrnaseq.metadata_MIC.csv \
    --count-files input/snrnaseq/protocol_example.snrnaseq.pseudobulk_counts_MIC.csv.gz \
    --tech-vars-file input/snrnaseq/protocol_example.snrnaseq.tech_vars_MIC.csv \
    --output-dir output/snrna_seq


## Step 3: Phenotype Reformatting (snATAC-seq only)

Converts residuals into a QTL-ready BED format for genome-wide caQTL mapping.

> For snRNA-seq, please follow this [pipeline](https://github.com/StatFunGen/xqtl-protocol/blob/main/code/data_preprocessing/phenotype/phenotype_formatting.ipynb).

### Input

| File | Description |
|------|-------------|
| `{celltype}_residuals.txt` | Residuals from `pseudobulk_qc` |

### Process

1. Read residuals file with proper handling of feature IDs and sample columns
2. Parse peak coordinates from peak IDs (`chr-start-end` format)
3. Convert to midpoint coordinates (standard for QTLtools):
```
start = floor((peak_start + peak_end) / 2)
end   = start + 1
```
4. Build BED format: `#chr`, `start`, `end`, `ID` followed by per-sample values
5. Sort by chromosome and position
6. Compress with `bgzip` and index with `tabix`

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `residual_files` | *required* | Residual txt files from `pseudobulk_qc` |
| `output_dir` | *required* | Parent output directory; writes to `{output_dir}/3_pheno_reformat/` |

### Output

Output directory: `{output_dir}/3_pheno_reformat/`

| File | Description |
|------|-------------|
| `{celltype}_phenotype.bed.gz` | bgzip-compressed BED with midpoint coordinates |
| `{celltype}_phenotype.bed.gz.tbi` | tabix index for random-access queries |

Compatible with FastQTL, TensorQTL, and QTLtools.

**Timing:** < 1 min per cell type

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb phenotype_formatting \
    --residual-files input/snrnaseq/atac_residuals/MIC/protocol_example.snrnaseq.MIC_residuals.txt \
    --output-dir output/snrna_seq \
    --gtf-file input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf


## Output Files

Written to `{output_dir}/2_residuals/{celltype}/` (e.g. `output/snrna_seq/2_residuals/MIC/`).

| File | Description |
|------|-------------|
| `{celltype}_residuals.txt` | Tech-covariate-adjusted values (log2-CPM) |
| `{celltype}_results.rds` | Full results: DGEList, fit, offset, residuals, design |
| `{celltype}_filtered_raw_counts.txt` | Filtered raw counts before normalization |

Stage 1 (`pseudobulk_counts`) writes `0_pseudobulk_counts/pseudobulk_counts_{celltype}.csv.gz`; `sampleid_mapping` writes `1_files_with_sampleid/`.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

## Command interface

In [ ]:
sos run pipeline/pseudobulk_preprocessing.ipynb -h

## Workflow implementation

The cells below define the SoS workflow steps invoked by the commands above. They are not meant to be edited for routine analysis.


## Setup and global parameters

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "5h"
parameter: mem = "16G"
parameter: numThreads = 8
parameter: container = ""


cwd = path(f'{cwd:a}')

```
usage: sos run pipeline/pseudobulk_preprocessing.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters
Workflows:
  sampleid_mapping
  pseudobulk_qc
  phenotype_formatting
Global Workflow Options:
  --cwd output (as path)
  --job-size 1 (as int)
  --walltime 5h
  --mem 16G
  --numThreads 8 (as int)
  --container ''
Sections
  sampleid_mapping:
    Workflow Options:
      --map-file VAL (as str, required)
      --output-dir VAL (as str, required)
      --meta-files  (as list)
      --count-files  (as list)
  pseudobulk_qc:
    Workflow Options:
      --meta-files  (as list)
      --count-files  (as list)
      --output-dir VAL (as str, required)
      --tech-vars-file VAL (as str, required)
      --blacklist-file ''
      --batch-correction FALSE
      --batch-method limma
      --quant-norm FALSE
      --min-count 5 (as int)
      --min-total-count 15 (as int)
      --min-prop 0.1 (as float)
      --min-nuclei 20 (as int)
      --regions ''
      --gene-list ''
  phenotype_formatting:
    Workflow Options:
      --residual-files  (as list)
      --output-dir VAL (as str, required)
      --gtf-file VAL (as path, required)
```

## `pseudobulk_counts`

In [ ]:
[pseudobulk_counts]
parameter: seurat_files = []
parameter: output_dir   = str
parameter: celltype     = 'MIC'
parameter: min_cells    = 10

import os

input:  seurat_files
output: f'{output_dir}/0_pseudobulk_counts/pseudobulk_counts_{celltype}.csv.gz'

task: trunk_workers = 1, trunk_size = 1, walltime = '4:00:00', mem = '64G', cores = 4

bash: expand = "${ }", stdout = f'{_output:n}.stdout', stderr = f'{_output:n}.stderr'
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/pseudobulk_preprocessing.R \
        --step pseudobulk_counts \
        --seurat-files ${' '.join([f'"{f}"' for f in seurat_files])} \
        --celltype "${celltype}" \
        --min-cells ${min_cells} \
        --output-dir "${output_dir}"


## `sampleid_mapping`

In [ ]:
[sampleid_mapping]
parameter: map_file    = str
parameter: output_dir  = str
parameter: meta_files  = []
parameter: count_files = []

import os

input:  meta_files + count_files
output: [f'{output_dir}/1_files_with_sampleid/{os.path.basename(f)}' for f in meta_files + count_files]
         
bash: expand = "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/pseudobulk_preprocessing.R \
        --step sampleid_mapping \
        --map-file "${map_file}" \
        ${('--meta-files ' + ' '.join([f'"{f}"' for f in meta_files])) if meta_files else ''} \
        ${('--count-files ' + ' '.join([f'"{f}"' for f in count_files])) if count_files else ''} \
        --output-dir "${output_dir}"


## `pseudobulk_qc`

In [ ]:
[pseudobulk_qc]
parameter: meta_files       = []
parameter: count_files      = []
parameter: output_dir       = str
parameter: tech_vars_file   = str
parameter: blacklist_file   = ''
parameter: batch_correction = "FALSE"
parameter: batch_method     = "limma"
parameter: quant_norm       = "FALSE"
parameter: min_count        = 5
parameter: min_total_count  = 15
parameter: min_prop         = 0.1
parameter: min_nuclei       = 20
parameter: regions          = ''
parameter: gene_list        = ''

import os

_cts = [os.path.basename(f).split('metadata_')[-1].replace('.csv','') for f in meta_files]

input:  meta_files + count_files
output: [f'{output_dir}/2_residuals/{ct}/{ct}_residuals.txt' for ct in _cts]

task: trunk_workers = 1, trunk_size = 1, walltime = '6:00:00', mem = '64G', cores = 4

bash: expand = "${ }", stdout = f'{_output[0]:n}.stdout', stderr = f'{_output[0]:n}.stderr'
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/pseudobulk_preprocessing.R \
        --step pseudobulk_qc \
        --meta-files ${' '.join([f'"{f}"' for f in meta_files])} \
        --count-files ${' '.join([f'"{f}"' for f in count_files])} \
        --tech-vars-file "${tech_vars_file}" \
        --blacklist-file "${blacklist_file}" \
        --batch-correction "${batch_correction}" \
        --batch-method "${batch_method}" \
        --quant-norm "${quant_norm}" \
        --min-count ${min_count} \
        --min-total-count ${min_total_count} \
        --min-prop ${min_prop} \
        --min-nuclei ${min_nuclei} \
        --regions "${regions}" \
        --gene-list "${gene_list}" \
        --output-dir "${output_dir}"


## `phenotype_reformatting`

In [ ]:
[phenotype_formatting]
parameter: residual_files = []
parameter: output_dir     = str
parameter: gtf_file       = path

import os

_cts = [os.path.basename(os.path.dirname(f)) for f in residual_files]

input:  residual_files
output: [f'{output_dir}/3_pheno_reformat/{ct}_phenotype.bed.gz' for ct in _cts]

task: trunk_workers = 1, trunk_size = 1, walltime = '2:00:00', mem = '16G', cores = 2

bash: expand = "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/pseudobulk_preprocessing.R \
        --step phenotype_formatting \
        --residual-files ${' '.join([f'"{f}"' for f in residual_files])} \
        --gtf-file "${gtf_file}" \
        --output-dir "${output_dir}"
